# Cafe — tối ưu dự đoán doanh thu trên Kaggle

**Mục tiêu:** dự đoán doanh thu ngày của từng cửa hàng. Chỉ số **RMSLE càng thấp càng tốt**.
Notebook này đã được chạy trên dữ liệu được cung cấp; các điểm dưới đây là **validation trên train, chưa phải điểm Kaggle**.
Chưa có cuộc thi hoặc `test.csv`, nên notebook lưu mô hình đầy đủ và tự bỏ qua phần xuất submission.

Trên Kaggle: tạo notebook → **Import Notebook** → chọn file này → **Add Input** bộ dữ liệu Cafe → **Run All**.
Chỉ cần CPU. Code tự tìm đúng bộ Cafe qua các file stores/weather và cột revenue,
tránh nhầm `train.csv` của bài Churn. Dữ liệu gốc được giữ nguyên.

In [ ]:
import os
os.environ.setdefault('OMP_NUM_THREADS', '4')
os.environ.setdefault('MPLCONFIGDIR', str(__import__('pathlib').Path.cwd() / '.matplotlib'))
import sys, json, platform, warnings, importlib.metadata
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', 30)
plt.rcParams.update({'figure.dpi': 115, 'axes.spines.top': False, 'axes.spines.right': False})
OUT = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd() / 'outputs'
OUT.mkdir(exist_ok=True)
print('Python:', platform.python_version())
for package in ['numpy','pandas','scikit-learn','catboost','lightgbm']:
    try: print(package, importlib.metadata.version(package))
    except importlib.metadata.PackageNotFoundError: print(package, '(chưa cài; chỉ cần nếu chọn mô hình này)')

## 1. Đọc dữ liệu và kiểm tra chất lượng

Các bảng được ghép bằng khóa nhiều-một; ngày có hai dịp lễ được gộp thành nhiều cờ
trước khi ghép để không nhân số dòng. Loại 132 dòng train trùng hoàn toàn.
Cần phân biệt những lỗi có bằng chứng với doanh thu cao/thấp có thể xảy ra thật.

In [ ]:
"""Forecast cafe revenue with chronological validation and no test-target leakage."""
from pathlib import Path
import json
import os
import time
import unicodedata
import joblib
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import Ridge

SEED = 2026
THREADS = min(6, os.cpu_count() or 2)


def rmsle(y, prediction):
    y = np.asarray(y, dtype=float)
    p = np.asarray(prediction, dtype=float)
    if not np.isfinite(y).all() or (y < 0).any():
        raise ValueError("Validation targets must be finite and non-negative")
    if not np.isfinite(p).all():
        raise ValueError("Predictions contain non-finite values")
    return float(np.sqrt(np.mean((np.log1p(y) - np.log1p(np.maximum(p, 0))) ** 2)))


def normalize_text(value):
    value = unicodedata.normalize("NFKD", str(value)).encode("ascii", "ignore").decode()
    return " ".join(value.lower().strip().split())


def read_data(folder):
    folder = Path(folder)
    tables = {name: pd.read_csv(folder / f"{name}.csv")
              for name in ["train", "stores", "weather", "holidays", "sample_submission"]}
    for name in ["train", "weather", "holidays"]:
        tables[name]["date"] = pd.to_datetime(tables[name]["date"], errors="raise")
    tr = tables["train"]
    if tr.revenue.isna().any() or (tr.revenue < 0).any():
        raise ValueError("Invalid target values: inspect before training")
    tables["train"] = tr.drop_duplicates().reset_index(drop=True)
    if tables["train"].duplicated(["date", "store_id"]).any():
        raise ValueError("Conflicting train rows for the same date/store")
    tables["test"] = None
    candidates = list(folder.rglob("test.csv"))
    if not candidates and Path("/kaggle/input").exists():
        candidates = list(Path("/kaggle/input").rglob("test.csv"))
    valid = []
    for path in candidates:
        header = pd.read_csv(path, nrows=0).columns
        if {"id", "date", "store_id", "is_promo"}.issubset(header):
            valid.append(path)
    if len(valid) > 1:
        raise ValueError(f"Multiple Cafe test files found; attach only one: {valid}")
    if valid:
        test = pd.read_csv(valid[0])
        test["date"] = pd.to_datetime(test.date, errors="raise")
        tables["test"] = test
    return tables


def describe_quality(tables):
    train, weather, holidays = (tables[k] for k in ["train", "weather", "holidays"])
    return {
        "train_unique_rows": len(train), "stores": len(tables["stores"]),
        "date_min": str(train.date.min().date()), "date_max": str(train.date.max().date()),
        "revenue_above_1000": int((train.revenue > 1000).sum()),
        "weather_fahrenheit_candidates": int((weather.temp_c > 55).sum()),
        "weather_missing_rain": int(weather.rain_mm.isna().sum()),
        "weather_missing_city_days": int(
            weather.city.nunique() * len(pd.date_range(weather.date.min(), weather.date.max())) - len(weather)),
        "holiday_dates_with_multiple_events": int(holidays.date.duplicated().sum()),
        "has_official_test": tables["test"] is not None,
    }


class FeatureBuilder:
    """Only exogenous data: stores, provided calendar and provided weather."""
    def __init__(self, tables):
        st = tables["stores"].copy()
        for col in ["city", "store_type", "district"]:
            st[col] = st[col].map(normalize_text)
        st["district"] = st.district.replace({"q.10": "district 10", "quan 3": "district 3", "quan 7": "district 7"})
        st["open_date"] = st.open_date.map(
            lambda s: pd.to_datetime(s, format="%d/%m/%Y" if "/" in s else "%Y-%m-%d"))
        st["area_missing"] = st.area_m2.isna().astype(int)
        # Physical size proxy from metadata, independent of any revenue target.
        st["area_m2"] = st.area_m2.fillna(st.seats * 2.5)
        if st.store_id.duplicated().any():
            raise ValueError("Store IDs must be unique")
        self.stores = st
        we = tables["weather"].copy()
        we["city"] = we.city.map(normalize_text)
        we.loc[we.temp_c > 55, "temp_c"] = (we.loc[we.temp_c > 55, "temp_c"] - 32) / 1.8
        if we.duplicated(["date", "city"]).any():
            raise ValueError("Weather keys must be unique")
        self.weather = we
        ho = tables["holidays"].copy()
        self.tet_dates = ho.loc[ho.holiday_name.eq("Lunar New Year (Tet)"), "date"].tolist()
        flags = pd.DataFrame({"date": ho.date})
        flags["national"] = ho.type.eq("national_holiday").astype(int)
        flags["tet_holiday"] = ho.holiday_name.str.contains("Tet holiday|Lunar New Year", regex=True).astype(int)
        events = {
            "new_year": "New Year's Day", "valentine": "Valentine's Day",
            "women_mar": "International Women's Day", "women_oct": "Vietnamese Women's Day",
            "teachers": "Teachers' Day", "xmas": "Christmas",
            "national_day": "National Day", "hung_kings": "Hung Kings",
            "reunification": "Reunification", "labour": "Labour", "kitchen_gods": "Ong Cong",
        }
        for name, pattern in events.items():
            flags[name] = ho.holiday_name.str.contains(pattern, regex=False).astype(int)
        self.holidays = flags.groupby("date", as_index=False).max()
        self.holiday_cols = list(flags.columns[1:])

    def transform(self, frame):
        df = frame.drop(columns=["revenue"], errors="ignore").copy()
        df["date"] = pd.to_datetime(df.date)
        count = len(df)
        df = df.merge(self.stores, on="store_id", how="left", validate="many_to_one")
        if df.city.isna().any():
            raise ValueError("Unknown store IDs")
        df = df.merge(self.weather, on=["date", "city"], how="left", validate="many_to_one")
        df = df.merge(self.holidays, on="date", how="left", validate="many_to_one")
        assert len(df) == count, "A join changed the number of prediction rows"
        df[self.holiday_cols] = df[self.holiday_cols].fillna(0)
        df["weather_missing"] = df.temp_c.isna().astype(int)
        df["rain_missing"] = df.rain_mm.isna().astype(int)
        df["temp_c"] = df.temp_c.fillna(26)
        df["rain_mm"] = df.rain_mm.fillna(0).clip(lower=0)
        date = df.date.dt
        df["dow"] = date.dayofweek.astype(str)
        df["month"] = date.month.astype(str)
        df["is_weekend"] = (date.dayofweek >= 5).astype(int)
        df["day_of_month"] = date.day
        df["day_of_year"] = date.dayofyear
        df["time_years"] = (df.date - pd.Timestamp("2024-01-01")).dt.days / 365.25
        df["age_days"] = (df.date - df.open_date).dt.days.clip(lower=0)
        df["log_age"] = np.log1p(df.age_days)
        for days in [14, 30, 60, 90, 180]:
            df[f"age_ramp_{days}"] = np.minimum(df.age_days / days, 1)
        for days in [3, 7, 14]:
            df[f"opening_{days}"] = (df.age_days < days).astype(int)
        for k in [1, 2, 3]:
            theta = 2 * np.pi * k * (date.dayofyear - 1) / 365.25
            df[f"sin_{k}"] = np.sin(theta)
            df[f"cos_{k}"] = np.cos(theta)
        df["temp_scaled"] = (df.temp_c - 26) / 10
        df["temp_square"] = df.temp_scaled ** 2
        df["rain_scaled"] = df.rain_mm / 10
        df["log_rain"] = np.log1p(df.rain_mm)
        df["rain_heavy"] = (df.rain_mm >= 20).astype(int)
        df["log_seats"] = np.log1p(df.seats)
        df["log_area"] = np.log1p(df.area_m2)
        df["type_dow"] = df.store_type + "_" + df.dow
        df["type_month"] = df.store_type + "_" + df.month
        df["store_dow"] = df.store_id + "_" + df.dow
        offsets = np.stack([(df.date - t).dt.days.to_numpy() for t in self.tet_dates])
        nearest = np.argmin(abs(offsets), axis=0)
        tet_offset = offsets[nearest, np.arange(len(df))]
        df["tet_offset"] = np.clip(tet_offset, -45, 45)
        df["tet_day"] = np.where(abs(tet_offset) <= 14, tet_offset.astype(str), "outside")
        df["type_tet"] = df.store_type + "_" + df.tet_day
        df["tet_before"] = ((tet_offset >= -14) & (tet_offset < -3)).astype(int)
        df["tet_core"] = ((tet_offset >= -3) & (tet_offset <= 4)).astype(int)
        df["tet_after"] = ((tet_offset >= 5) & (tet_offset <= 14)).astype(int)
        df["national_other"] = df.national * (1 - df.tet_holiday)
        df["tet_fixed_3"] = ((tet_offset >= -2) & (tet_offset <= 3)).astype(int)
        df["tet_fixed_4"] = ((tet_offset >= -2) & (tet_offset <= 4)).astype(int)
        df["summer"] = date.month.isin([6, 7, 8]).astype(int)
        df["winter"] = date.month.isin([1, 2]).astype(int)
        df["near_christmas"] = ((date.month == 12) & date.day.between(20, 26)).astype(int)
        df["id_source_index"] = np.arange(len(df))
        return df


def clean_training_target(frame, fix_units=True):
    """Training only: a discontinuous x1000 block; never alter validation labels."""
    y = frame.revenue.to_numpy(float).copy()
    if fix_units:
        y[y > 1000] /= 1000
    return y


def linear_records(features, extra=False):
    if extra in [5, 6, 7, 8, 9, 10]:
        features = features.copy()
        fixed = features["tet_fixed_3" if extra in [5, 7, 9] else "tet_fixed_4"]
        features["tet_holiday"] = fixed
        features["national"] = np.maximum(features.national_other, fixed)
    cats = ["store_id", "city", "store_type", "district", "type_dow", "dow", "month"]
    nums = ["log_seats", "log_area", "has_parking", "is_promo", "time_years",
            "temp_scaled", "temp_square", "rain_scaled", "log_rain", "rain_heavy",
            "age_ramp_14", "age_ramp_30", "age_ramp_60", "age_ramp_90", "age_ramp_180",
            "national", "tet_holiday", "tet_before", "tet_core", "tet_after",
            "new_year", "valentine", "women_mar", "women_oct", "teachers", "xmas",
            "national_day", "hung_kings", "reunification", "labour", "kitchen_gods",
            "sin_1", "cos_1", "sin_2", "cos_2", "sin_3", "cos_3", "summer", "winter"]
    if extra:
        cats += ["store_dow", "type_month", "type_tet"]
    advanced = extra in [3, 4]
    if advanced:
        # The supplied Tet holiday span changes between years. Learn lunar-day
        # effects, rather than assuming that an added holiday is a closure day.
        nums = [n for n in nums if n not in ["national", "tet_holiday", "tet_before", "tet_core", "tet_after"]]
        nums += ["national_other", "opening_3", "opening_7", "opening_14"]
    elif extra in [9, 10]:
        nums += ["opening_3", "opening_7", "opening_14"]
    subset = features[cats + nums].copy()
    for col in cats:
        subset[col] = subset[col].astype(str)
    interaction_nums = ["is_promo", "time_years", "national", "tet_holiday", "tet_before",
                        "tet_core", "tet_after", "new_year", "valentine", "women_mar",
                        "women_oct", "teachers", "xmas", "national_day", "hung_kings",
                        "reunification", "labour", "summer", "winter", "rain_scaled",
                        "log_rain", "temp_scaled", "temp_square", "sin_1", "cos_1", "sin_2", "cos_2"]
    if advanced:
        interaction_nums = [n for n in interaction_nums if n not in ["national", "tet_holiday", "tet_before", "tet_core", "tet_after"]]
        interaction_nums += ["national_other", "opening_3", "opening_7"]
    elif extra in [9, 10]:
        interaction_nums += ["opening_3", "opening_7"]
    # Column-wise interactions avoid a slow Python loop over dataframe rows.
    expanded = {}
    for typ in sorted(features.store_type.unique()):
        mask = features.store_type.eq(typ).astype(float)
        for col in interaction_nums:
            expanded[f"type:{typ}:{col}"] = features[col] * mask
    for city in sorted(features.city.unique()):
        mask = features.city.eq(city).astype(float)
        for col in ["temp_scaled", "rain_scaled", "sin_1", "cos_1"]:
            expanded[f"city:{city}:{col}"] = features[col] * mask
    if extra is True or extra in [1, 4, 7, 8, 9, 10]:
        for store in sorted(features.store_id.unique()):
            expanded[f"store:{store}:time_years"] = features.time_years * features.store_id.eq(store)
    subset = pd.concat([subset, pd.DataFrame(expanded, index=features.index)], axis=1)
    records = subset.to_dict("records")
    # Keep the matrix sparse, including in cold-start prediction.
    return [{key: val for key, val in row.items() if isinstance(val, str) or val != 0}
            for row in records]


class AdditiveModel:
    def __init__(self, alpha=3.0, robust=None, extra=False, log_target="log1p"):
        self.alpha, self.robust, self.extra, self.log_target = alpha, robust, extra, log_target

    def fit(self, features, y):
        self.vectorizer = DictVectorizer(sparse=True)
        X = self.vectorizer.fit_transform(linear_records(features, self.extra))
        z = np.log1p(y) if self.log_target == "log1p" else np.log(np.maximum(y, 0.01))
        weights = np.ones(len(y))
        for _ in range(8 if self.robust else 1):
            self.model = Ridge(alpha=self.alpha, solver="lsqr", tol=1e-6)
            self.model.fit(X, z, sample_weight=weights)
            if self.robust:
                residual = z - self.model.predict(X)
                weights = np.minimum(1, self.robust / np.maximum(abs(residual), 1e-8))
        return self

    def predict_log(self, features):
        z = self.model.predict(self.vectorizer.transform(linear_records(features, self.extra)))
        if self.log_target == "log":
            z = np.log1p(np.exp(np.clip(z, -15, 15)))
        return np.maximum(z, 0)

    def predict(self, features):
        return np.expm1(self.predict_log(features))


def tree_features(features):
    drop = ["date", "open_date", "id", "id_source_index"]
    X = features.drop(columns=drop, errors="ignore").copy()
    # pandas 2 rejects select_dtypes(include=['str']); this supports 2.x and 3.x.
    cats = [col for col in X.columns if pd.api.types.is_object_dtype(X[col].dtype)
            or pd.api.types.is_string_dtype(X[col].dtype)]
    for col in cats:
        X[col] = X[col].astype(str).fillna("missing")
    X = X.replace([np.inf, -np.inf], np.nan)
    return X, cats


def weekday_median(history, target):
    h = history.assign(dow=history.date.dt.dayofweek)
    m = h.groupby(["store_id", "dow"]).revenue.median()
    keys = list(zip(target.store_id, target.date.dt.dayofweek))
    p = pd.Series(keys).map(m)
    p = p.fillna(target.store_id.map(h.groupby("store_id").revenue.median()).reset_index(drop=True))
    return p.fillna(h.revenue.median()).to_numpy()


FOLDS = [
    ("2025_Q1", "2025-01-01", "2025-04-01"),
    ("2025_Q2", "2025-04-01", "2025-07-01"),
    ("2025_Q3", "2025-07-01", "2025-10-01"),
    ("2025_Q4", "2025-10-01", "2026-01-01"),
]


def save_submission(test, sample, predictions, output_path):
    if test is None:
        raise ValueError("Official test.csv is required; sample_submission is only a template")
    if test.id.duplicated().any() or sample.id.duplicated().any():
        raise ValueError("Duplicate submission IDs")
    if len(test) != len(sample) or set(test.id) != set(sample.id):
        raise ValueError("Official test IDs do not match sample_submission")
    result = pd.DataFrame({"id": test.id, "revenue": np.maximum(predictions, 0)})
    result = sample[["id"]].merge(result, on="id", how="left", validate="one_to_one")
    if not np.isfinite(result.revenue).all():
        raise ValueError("Submission contains invalid predictions")
    result.to_csv(output_path, index=False)
    return result

In [ ]:
candidates = []
for base in [Path('/kaggle/input'), Path.cwd(), Path.cwd() / 'cafe']:
    if not base.exists(): continue
    hits = list(base.rglob('stores.csv')) if str(base) == '/kaggle/input' else [base / 'stores.csv']
    for path in hits:
        folder = path.parent
        if all((folder / f'{n}.csv').exists() for n in ['train','weather','holidays','sample_submission']):
            header = pd.read_csv(folder / 'train.csv', nrows=0).columns
            if {'revenue','store_id','date','is_promo'}.issubset(header): candidates.append(folder.resolve())
candidates = sorted(set(candidates))
if len(candidates) != 1:
    raise ValueError(f'Cần đúng một bộ dữ liệu Cafe. Tìm thấy: {candidates}')
DATA = candidates[0]
tables = read_data(DATA)
train = tables['train']
quality = describe_quality(tables)
raw_rows = len(pd.read_csv(DATA / 'train.csv'))
quality['exact_duplicates_removed'] = raw_rows - len(train)
print('Data:', DATA)
display(pd.DataFrame([quality]).T.rename(columns={0:'value'}))
display(train.head())
display(tables['stores'].head())
_ = (OUT / 'data_quality.json').write_text(json.dumps(quality, indent=2), encoding='utf-8')

## 2. Khám phá dữ liệu và sửa lỗi có bằng chứng

- S12 tháng 3/2024 có 30 doanh thu trên 1.000; trung vị tháng khoảng 29.165,
  trong khi các tháng xung quanh khoảng 25–31. Chia các giá trị này cho 1.000 **chỉ khi huấn luyện**.
- 22 nhiệt độ khoảng 82–89 trong cột Celsius tương ứng khoảng 28–32°C sau đổi Fahrenheit.
- Chuẩn hóa kiểu cửa hàng, tên quận và hai định dạng ngày khai trương.
- Giữ nguyên các nhãn validation, kể cả doanh thu tăng/giảm đột biến. Không xóa điểm khó để làm đẹp số.
- Với dữ liệu thiếu, tạo cờ thiếu; phần mô hình chỉ học từ lịch sử trước mốc kiểm tra.

In [ ]:
builder = FeatureBuilder(tables)
features = builder.transform(train)
cleaned_target = clean_training_target(train)
s12 = train.loc[train.store_id.eq('S12')].copy()
s12['month'] = s12.date.dt.to_period('M').astype(str)
s12['corrected'] = clean_training_target(s12)
display(s12.groupby('month')[['revenue','corrected']].median().head(6))
plotdata = features[['date','store_id','store_type','dow']].copy()
plotdata['revenue'] = cleaned_target
plotdata['month'] = plotdata.date.dt.to_period('M').astype(str)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
plotdata.groupby('month').revenue.median().plot(ax=axes[0], marker='o', color='#166e75')
axes[0].set(title='Monthly median revenue after unit correction', ylabel='Million VND', xlabel='')
axes[0].tick_params(axis='x', rotation=65)
dow_table = plotdata.groupby(['dow','store_type']).revenue.median().unstack()
dow_table.plot(ax=axes[1], marker='o')
axes[1].set(title='Weekday patterns differ by store type', xlabel='Weekday: Monday=0', ylabel='Million VND')
fig.tight_layout()
fig.savefig(OUT / 'data_overview.png', bbox_inches='tight')
plt.show()

## 3. Thiết kế validation và mô hình

Dùng 4 quý năm 2025. Mỗi lần chỉ học các ngày **trước quý đang kiểm tra**,
rồi dự đoán toàn bộ quý mà không nhìn doanh thu trong quý đó. Không dùng random split,
lag lấy từ doanh thu tương lai, hoặc test giả dựng từ sample submission.

Thử mô hình trung vị, Ridge có tương tác theo cửa hàng/lịch/thời tiết,
CatBoost và LightGBM. Mô hình được chọn dựa trên nhiều quý, có chú ý quý I
vì cùng mùa với giai đoạn cần dự đoán. Hiệu ứng Tết dựa vào khoảng cách ngày đến Tết;
không mặc định mọi ngày được ghi nghỉ Tết đều có mức đóng cửa giống nhau.

Các tham số/khối đặc trưng đã được tinh chỉnh bằng validation; các điểm này có thể lạc quan
hơn một tập kiểm tra hoàn toàn mới. Kết quả cuộc thi chỉ xác nhận được sau khi có test và leaderboard.

In [ ]:
CONFIG = {'selection': 'Four chronological 2025 quarter folds; favor stable performance and Q1 seasonality. No Kaggle test or leaderboard available.', 'models': [{'name': 'seasonal_ridge', 'kind': 'additive', 'alpha': 3, 'robust': 0.3, 'extra': 9, 'log_target': 'log1p', 'weight': 0.7}, {'name': 'seasonal_catboost', 'kind': 'residual', 'base': {'alpha': 3, 'robust': 0.3, 'extra': 7, 'log_target': 'log1p'}, 'iterations': 400, 'depth': 5, 'loss': 'RMSE', 'weight': 0.3}]}
print(json.dumps(CONFIG, indent=2, ensure_ascii=False))

In [ ]:
"""Selected-model validation, final fitting, and forecasting."""


class ResidualModel:
    def __init__(self, spec):
        self.spec = spec

    def fit(self, features, y):
        from catboost import CatBoostRegressor
        self.base = AdditiveModel(**self.spec['base']).fit(features, y)
        residual = np.log1p(y) - self.base.predict_log(features)
        X, cats = tree_features(features)
        self.tree = CatBoostRegressor(iterations=self.spec['iterations'], depth=self.spec.get('depth', 5),
            learning_rate=.035, loss_function=self.spec.get('loss', 'RMSE'), l2_leaf_reg=15,
            random_seed=SEED, thread_count=THREADS, allow_writing_files=False, verbose=False)
        self.tree.fit(X, residual, cat_features=cats)
        return self

    def predict_log(self, features):
        X, _ = tree_features(features)
        return np.maximum(self.base.predict_log(features) + self.tree.predict(X), 0)


def make_selected_model(spec, seed=SEED):
    if spec['kind'] == 'residual':
        return ResidualModel(spec)
    if spec['kind'] == 'additive':
        return AdditiveModel(alpha=spec['alpha'], robust=spec.get('robust'),
                             extra=spec.get('extra', False), log_target=spec.get('log_target', 'log1p'))
    if spec['kind'] == 'catboost':
        from catboost import CatBoostRegressor
        return CatBoostRegressor(iterations=spec['iterations'], depth=spec.get('depth', 7),
            learning_rate=spec.get('learning_rate', .04), loss_function='RMSE', l2_leaf_reg=8,
            random_seed=seed, thread_count=THREADS, allow_writing_files=False, verbose=False)
    raise ValueError(f'Unknown model kind: {spec}')


def fit_one(spec, features, y):
    model = make_selected_model(spec)
    if spec['kind'] in ['additive', 'residual']:
        model.fit(features, y)
    else:
        X, cats = tree_features(features)
        model.fit(X, np.log1p(y), cat_features=cats)
    return model


def predict_one(spec, model, features):
    if spec['kind'] in ['additive', 'residual']:
        return model.predict_log(features)
    X, _ = tree_features(features)
    return np.maximum(model.predict(X), 0)


def validate_selected(tables, builder, features, config, out):
    weights = [s['weight'] for s in config['models']]
    if min(weights) < 0 or not np.isclose(sum(weights), 1):
        raise ValueError('Ensemble weights must be non-negative and sum to one')
    train = tables['train']
    rows, parts = [], []
    started = time.monotonic()
    for fold, start, stop in FOLDS:
        fi = train.date < start
        vi = (train.date >= start) & (train.date < stop)
        history, valid = train.loc[fi], train.loc[vi]
        y = clean_training_target(history)
        z = np.log1p(valid.revenue.to_numpy())
        part = valid[['date','store_id','revenue']].copy()
        part['fold'] = fold
        part['seen_store'] = valid.store_id.isin(history.store_id.unique())
        part['baseline'] = np.log1p(weekday_median(history, valid))
        for spec in config['models']:
            model = fit_one(spec, features.loc[fi], y)
            part[spec['name']] = predict_one(spec, model, features.loc[vi])
        part['ensemble'] = sum(spec['weight'] * part[spec['name']] for spec in config['models'])
        for name in ['baseline'] + [s['name'] for s in config['models']] + ['ensemble']:
            score = rmsle(valid.revenue, np.expm1(part[name]))
            rows.append({'fold':fold, 'model':name, 'rmsle':score, 'rows':len(valid)})
            print(f'{fold} {name}: {score:.6f}', flush=True)
        parts.append(part)
        pd.DataFrame(rows).to_csv(out / 'selected_cv_metrics.csv', index=False)
        pd.concat(parts).to_csv(out / 'selected_cv_predictions.csv', index=False)
    metrics, predictions = pd.DataFrame(rows), pd.concat(parts, ignore_index=True)
    pooled = {name:rmsle(predictions.revenue, np.expm1(predictions[name])) for name in ['baseline','ensemble']}
    metadata = {'validation':'Expanding-window 2025 quarters; raw validation targets; tuned configuration',
                'pooled_rmsle':pooled, 'mean_quarter_rmsle':metrics.groupby('model').rmsle.mean().to_dict(),
                'has_kaggle_score':False, 'train_rows':len(train), 'configuration':config,
                'elapsed_seconds':round(time.monotonic()-started, 2)}
    (out / 'metrics.json').write_text(json.dumps(metadata, indent=2, ensure_ascii=False), encoding='utf-8')
    return metrics, predictions


def fit_final(tables, builder, features, config, out):
    y = clean_training_target(tables['train'])
    models = []
    for spec in config['models']:
        started = time.monotonic()
        model = fit_one(spec, features, y)
        models.append((spec, model))
        print(f'Full train: {spec["name"]}; {len(y)} rows; {time.monotonic()-started:.1f}s', flush=True)
    trained = {'models':models, 'feature_builder':builder,
               'train_last_date':str(tables['train'].date.max().date()), 'seed':SEED}
    joblib.dump(trained, out / 'cafe_final.joblib', compress=3)
    return trained


def predict_final(trained, features):
    total = sum(spec['weight'] * predict_one(spec, model, features) for spec, model in trained['models'])
    if not np.isfinite(total).all():
        raise ValueError('Non-finite forecasts')
    return np.expm1(np.maximum(total, 0))

In [ ]:
cv_rows, cv_predictions = validate_selected(tables, builder, features, CONFIG, OUT)
scores = cv_rows.pivot(index='model', columns='fold', values='rmsle')
scores['mean_quarters'] = scores.mean(axis=1)
display(scores.round(6))
print('Pooled RMSLE (all held-out rows):')
for name in ['baseline', 'ensemble']:
    print(name, rmsle(cv_predictions.revenue, np.expm1(cv_predictions[name])))

## 4. So sánh và phân tích lỗi

Bảng tính trên **toàn bộ** các dòng validation, gồm cả cửa hàng chưa xuất hiện trong lịch sử.
Phân tích lỗi theo loại cửa hàng, tháng và khả năng cửa hàng đã có lịch sử để tìm điểm yếu.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
scores.drop(columns='mean_quarters').T.plot.bar(ax=ax)
ax.set(title='Chronological validation — lower is better', ylabel='RMSLE', xlabel='')
ax.tick_params(axis='x', rotation=0)
ax.legend(loc='upper left', bbox_to_anchor=(1, 1))
fig.tight_layout()
fig.savefig(OUT / 'validation_scores.png', bbox_inches='tight')
plt.show()
analysis = cv_predictions.merge(
    builder.stores[['store_id','store_type','city']], on='store_id', how='left', validate='many_to_one')
analysis['error_log'] = np.log1p(analysis.revenue) - analysis.ensemble
group_scores = analysis.groupby(['fold','store_type']).error_log.agg(
    count='size', rmsle=lambda s: np.sqrt(np.mean(s**2)), bias='mean')
display(group_scores.round(5))
group_scores.to_csv(OUT / 'error_by_store_type.csv')
display(analysis.groupby(['fold','seen_store']).error_log.agg(
    count='size', rmsle=lambda s: np.sqrt(np.mean(s**2))).round(5))
display(analysis.assign(abs_error=analysis.error_log.abs()).nlargest(12,'abs_error')[
    ['date','store_id','revenue','error_log']])

## 5. Train lại trên toàn bộ lịch sử và lưu mô hình

Sau khi chốt tham số, train lại trên toàn bộ dữ liệu đến 31/12/2025.
Điểm train không dùng để tuyên bố chất lượng dự đoán. File mô hình dùng lại khi có test chính thức.

In [ ]:
trained = fit_final(tables, builder, features, CONFIG, OUT)
print('Đã lưu:', OUT / 'cafe_final.joblib')
print('Số dòng huấn luyện:', len(train))

## 6. Xuất submission khi có test.csv chính thức

`sample_submission.csv` chỉ cung cấp ID và cấu trúc file, không có ngày/cửa hàng/khuyến mãi.
Khi chưa có test, phần này không tạo submission để tránh file dự đoán sai.
Sau này gắn thêm test.csv có các cột `id, date, store_id, is_promo`, chạy lại notebook.
Code kiểm tra đủ ID, đúng thứ tự sample, giá trị hữu hạn và doanh thu không âm.

In [ ]:
if tables['test'] is None:
    print('CHƯA CÓ TEST CHÍNH THỨC — đã train và lưu mô hình; chưa tạo submission.csv.')
else:
    test_features = builder.transform(tables['test'])
    prediction = predict_final(trained, test_features)
    submission = save_submission(tables['test'], tables['sample_submission'], prediction, OUT / 'submission.csv')
    print('Submission:', OUT / 'submission.csv', submission.shape)
    display(submission.head())

## Tài liệu và các lần chạy

Chỉ dùng dữ liệu Cafe được cung cấp. Nguồn tham khảo kỹ thuật:
[CatBoost regression objectives](https://catboost.ai/docs/en/concepts/loss-functions-regression),
[LightGBM parameters](https://lightgbm.readthedocs.io/en/stable/Parameters.html).

Các thử nghiệm rộng hơn được lưu riêng trong `outputs/experiment_metrics.csv`,
`outputs/additive_metrics.csv`, `outputs/calendar_metrics.csv`.
Chưa có cuộc thi nên không thể xác nhận hay bảo đảm thứ hạng cao nhất.